<a href="https://colab.research.google.com/github/NazmulHudaNabil/email-spam-detection/blob/main/Spam_or_not_using_Transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### For log Save

In [ ]:
# Force tqdm to use a simple text-based progress bar
import tqdm
def fixed_tqdm(*args, **kwargs):
    kwargs['gui'] = False
    return tqdm.tqdm(*args, **kwargs)

# If using Hugging Face Transformers, disable the fancy notebook widgets
import os
os.environ["WANDB_DISABLED"] = "true" # Optional: if you aren't using WandB

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import torch

In [ ]:
df = pd.read_csv("/content/spam.csv")
df.head()

In [ ]:
df.shape

In [ ]:
df.duplicated().sum()

In [ ]:
df.drop_duplicates(inplace=True)

### Target Value Lable Encoding

In [ ]:
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()
y = encoder.fit_transform(df['Category'])

In [ ]:
y

### Data Train Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(df['Message'], y, test_size=0.1, random_state=42)

In [ ]:
X_train.shape, X_test.shape

### Create pytorch CustomDataset for Transformer

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

# ---------------------------------------------------------
# 1. Define your custom dataset
# ---------------------------------------------------------
class MyTextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts.iloc[idx]) # Use .iloc for positional indexing
        label = self.labels[idx]

        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )


        # Squeeze to remove batch dimension added by return_tensors="pt"
        item = {key: val.squeeze(0) for key, val in encoding.items()}
        item["labels"] = torch.tensor(label, dtype=torch.long)
        return item

### Transformer Fine tuning "roberta-base" becasue it's good for Binary Classifcation

In [ ]:
from transformers import RobertaTokenizer, RobertaForSequenceClassification
tokenizer = RobertaTokenizer.from_pretrained('roberta-base')
model = RobertaForSequenceClassification.from_pretrained('roberta-base', num_labels=2)

In [ ]:
train_dataset = MyTextDataset(X_train, y_train, tokenizer)
test_dataset = MyTextDataset(X_test, y_test, tokenizer)

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

optimizer = AdamW(model.parameters(), lr=5e-5)
model.train()

### Transformer Model training

In [ ]:
epochs = 5
for epoch in range(epochs):
    total_loss = 0

    for batch in train_loader:
        # Move batch to device
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        # Forward pass
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        loss = outputs.loss

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch + 1}/{epochs} | Loss: {avg_loss:.4f}")

### Evalute the Test Data

In [ ]:
# Set the model to evaluation mode
model.eval()

correct_predictions = 0
total_predictions = 0

with torch.no_grad(): # Disable gradient calculation during evaluation
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits

        # Get predicted labels
        _, predicted = torch.max(logits, 1)

        total_predictions += labels.size(0)
        correct_predictions += (predicted == labels).sum().item()

accuracy = correct_predictions / total_predictions
print(f"Test Accuracy: {accuracy:.4f}")

#### Find model accuracy on test data is 99.61% it's very good

## Save this Model

In [ ]:
save_path = "./my_model"

model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

## Load the Model

In [ ]:
import os
from transformers import RobertaTokenizer, RobertaForSequenceClassification

# Use absolute path
save_path = "./my_model" # The model and tokenizer were saved directly to /content

tokenizer = RobertaTokenizer.from_pretrained(save_path)
model = RobertaForSequenceClassification.from_pretrained(save_path)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print("Loaded successfully!")

## After the load the model again check Model accucay on test data becasuse to check Save & load worked correctly or not

In [ ]:
# Set the model to evaluation mode
model.eval()

correct_predictions = 0
total_predictions = 0

with torch.no_grad(): # Disable gradient calculation during evaluation
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits

        # Get predicted labels
        _, predicted = torch.max(logits, 1)

        total_predictions += labels.size(0)
        correct_predictions += (predicted == labels).sum().item()

accuracy = correct_predictions / total_predictions
print(f"Test Accuracy: {accuracy:.4f}")

### Testing How it's work

In [ ]:
import torch

model.eval()

text = input("Enter Your text: ")
inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)

# Move input tensors to the same device as the model
inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits
    predicted_class = torch.argmax(logits, dim=1).item()

spam_or_ham = "spam" if predicted_class == 1 else "ham"
print(f"Predicted class: {spam_or_ham}")